In [1]:
import pandas as pd
import json
import os
import gradio as gr

DATA_PATH = './rank_dataset_98k.csv'
VOCAB_PATH = '../gym_model/vocab.json'

print("📥 Загрузка основного датасета...")
rank_df = pd.read_csv(DATA_PATH)

print("📥 Загрузка словаря упражнений...")
if os.path.exists(VOCAB_PATH):
    with open(VOCAB_PATH, 'r') as f:
        vocab = json.load(f)
    id2name = {v: k for k, v in vocab.items()}
    print(f"✅ Словарь загружен! Всего упражнений в словаре: {len(vocab)}")
else:
    print(f"❌ ОШИБКА: Файл {VOCAB_PATH} не найден!")

/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Загрузка основного датасета...
📥 Загрузка словаря упражнений...
✅ Словарь загружен! Всего упражнений в словаре: 3058


In [ ]:
import pandas as pd

# Предполагается, что rank_df и id2name у тебя уже загружены в памяти
unique_ids = rank_df['candidate_id'].dropna().unique()

exercise_list = []
for ex_id in unique_ids:
    name = id2name.get(ex_id, "Unknown")
    exercise_list.append({'candidate_id': ex_id, 'exercise_name': name})

df_unique_ex = pd.DataFrame(exercise_list)

def draft_equipment_manual_control(name):
    name = str(name).lower()

    # 1. ТРЕНАЖЕРЫ И БЛОКИ
    machine_keys = [
        'machine', 'cable', 'smith', 'leg press', 'pulldown', 'pec deck', 
        'leg extension', 'tricep extension', 'pushdown', 'row (cable)', 'fly (cable)', 
        'assisted', 'hack squat', 't-bar', 'sled', 'pulley', 'trx', 
        'calf block', 'glute drive', 'lat prayer', 'reverse hyper', 'belt squat',
        'abductor', 'adductor', 'leg curl', 'pit shark', 'v-squat'
    ]
    if any(k in name for k in machine_keys):
        return 'Machine'
        
    # 2. ГАНТЕЛИ И ГИРИ (ДО ШТАНГИ)
    dumbbell_keys = [
        'dumbbell', '(db)', 'kettlebell', '(kb)', 'lateral raise', 
        'hammer curl', 'goblet', 'renegade row', 'concentration curl',
        'arnold press', 'cuban press', 'halo', 'turkish get up'
    ]
    if any(k in name for k in dumbbell_keys):
        return 'Dumbbell'
        
    # 3. ШТАНГА
    barbell_keys = [
        'barbell', '(bb)', 'snatch', 'clean', 'front squat', 'back squat', 
        'thruster', 'good morning', 'rack pull', 'zercher', 'split jerk',
        'pendlay', 'landmine', 'box squat', 'pin squat', 'floor press', 'romanian',
        'trap bar', 'hex bar', 'ssb', 'safety squat', 'cambered'
    ]
    if any(k in name for k in barbell_keys) or any(base in name for base in ['deadlift', 'bench press', 'overhead press']):
        return 'Barbell'

    # 4. КРИСТАЛЬНО ЧИСТЫЙ СВОЙ ВЕС
    bodyweight_keys = [
        'bodyweight', 'body weight', 'push up', 'push-up', 'pull up', 'pull-up', 
        'chin up', 'plank', 'crunch', 'sit up', 'sit-up', 'dip', 'burpee', 
        'jump', 'wall sit', 'mountain climber', 'superman', 'hollow body', 
        'v-up', 'leg raise', 'pistol squat', 'muscle up', 'handstand', 
        'air squat', 'bear crawl', 'bird dog', 'bicycle', 'dead bug', 
        'flutter kick', 'scissor kick'
    ]
    if any(k in name for k in bodyweight_keys):
        return 'Bodyweight'

    # 5. ВСЁ ОСТАЛЬНОЕ ИДЕТ К ТЕБЕ (включая кардио, резину, валики, squat, lunge и т.д.)
    return 'REVIEW_NEEDED'

# Применяем функцию
df_unique_ex['equipment_golden'] = df_unique_ex['exercise_name'].apply(draft_equipment_manual_control)

# Сохраняем файл
export_path = 'exercises_to_review_MANUAL.csv'
df_unique_ex.sort_values(by='equipment_golden', ascending=False).to_csv(export_path, index=False, encoding='utf-8-sig')

print(f"✅ Файл '{export_path}' успешно создан!")
print(f"Всего уникальных упражнений: {len(df_unique_ex)}\n")
print("📊 Распределение (Всё спорное ушло в REVIEW_NEEDED):")
print(df_unique_ex['equipment_golden'].value_counts())

In [ ]:
import gradio as gr
import pandas as pd
import os
from deep_translator import GoogleTranslator

MAIN_FILE = 'exercises_to_review_MANUAL.csv'
PROGRESS_FILE = 'reviewed_exercises.csv' 

def get_next_exercise():
    df_main = pd.read_csv(MAIN_FILE)
    total_to_review = len(df_main[df_main['equipment_golden'] == 'REVIEW_NEEDED'])
    
    # Считаем статистику по категориям
    cat_stats = ""
    if os.path.exists(PROGRESS_FILE):
        df_prog = pd.read_csv(PROGRESS_FILE)
        reviewed_ids = set(df_prog['candidate_id'])
        
        # Группируем для счетчиков
        counts = df_prog['equipment_golden'].value_counts()
        
        # Формируем строку со счетчиками
        stats_list = [
            f"🤸‍♂️ Вес: {counts.get('Bodyweight', 0)}",
            f"🏋️‍♂️ Штанга: {counts.get('Barbell', 0)}",
            f"🦾 Гантель: {counts.get('Dumbbell', 0)}",
            f"⚙️ Маш: {counts.get('Machine', 0)}",
            f"🏃‍♂️ Кардио: {counts.get('Cardio', 0)}",
            f"🎗️ Резина: {counts.get('Bands', 0)}",
            f"🗑️ Другое: {counts.get('Other', 0)}"
        ]
        cat_stats = " | ".join(stats_list)
    else:
        reviewed_ids = set()
        cat_stats = "Пока нет данных"
        
    mask = (df_main['equipment_golden'] == 'REVIEW_NEEDED') & (~df_main['candidate_id'].isin(reviewed_ids))
    remaining_df = df_main[mask]
    
    remaining = len(remaining_df)
    done = total_to_review - remaining
    
    # Главная полоска прогресса + детальные счетчики
    main_stats = f"### 📊 Размечено: **{done}** | ⏳ Осталось: **{remaining}** из {total_to_review}"
    detailed_stats = f"**Распределение:** {cat_stats}"
    
    if remaining == 0:
        return None, "🎉 ВСЕ УПРАЖНЕНИЯ РАЗМЕЧЕНЫ! 🎉", main_stats, detailed_stats, ""
    
    cand_id = remaining_df.iloc[0]['candidate_id']
    ex_name = remaining_df.iloc[0]['exercise_name']
    
    # Bing Images iframe
    search_url = f"https://www.bing.com/images/search?q={ex_name.replace(' ', '+')}&adlt=off"
    iframe_html = f'''
    <div style="width: 100%; height: 500px; overflow: hidden; border: 2px solid #ddd; border-radius: 10px;">
        <iframe src="{search_url}" width="100%" height="600px" style="margin-top: -100px; border: none;"></iframe>
    </div>
    '''
    
    try:
        translated_name = GoogleTranslator(source='en', target='ru').translate(ex_name)
        display_text = f"{ex_name}\n🇷🇺 {translated_name}"
    except:
        display_text = f"{ex_name}\n🇷🇺 (Ошибка перевода)"
        
    return cand_id, display_text, main_stats, detailed_stats, iframe_html

def process_choice(choice, cand_id):
    if cand_id is not None:
        new_record = pd.DataFrame([{'candidate_id': cand_id, 'equipment_golden': choice}])
        if os.path.exists(PROGRESS_FILE):
            df_prog = pd.read_csv(PROGRESS_FILE).drop_duplicates()
            df_prog = pd.concat([df_prog, new_record], ignore_index=True)
        else:
            df_prog = new_record
        df_prog.to_csv(PROGRESS_FILE, index=False, encoding='utf-8-sig')
    return get_next_exercise()

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏋️‍♂️ Разметка с расширенной статистикой")
    
    with gr.Column():
        stats_display = gr.Markdown() # Общий прогресс
        detailed_stats_display = gr.Markdown() # Счетчики категорий
    
    current_id = gr.State()
    
    with gr.Row():
        exercise_display = gr.Textbox(label="Упражнение", interactive=False, lines=2)
    
    image_window = gr.HTML()
        
    with gr.Row():
        btn_bw = gr.Button("🤸‍♂️ Вес тела", variant="primary")
        btn_bb = gr.Button("🏋️‍♂️ Штанга", variant="primary")
        btn_db = gr.Button("🦾 Гантель", variant="primary")
        btn_mach = gr.Button("⚙️ Машина", variant="primary")
    
    with gr.Row():
        btn_cardio = gr.Button("🏃‍♂️ Кардио", variant="secondary")
        btn_bands = gr.Button("🎗️ Резина", variant="secondary")
        btn_other = gr.Button("🗑️ Другое", variant="stop")
        
    demo.load(fn=get_next_exercise, inputs=None, 
              outputs=[current_id, exercise_display, stats_display, detailed_stats_display, image_window])
        
    btns = [btn_bw, btn_bb, btn_db, btn_mach, btn_cardio, btn_bands, btn_other]
    choices = ['Bodyweight', 'Barbell', 'Dumbbell', 'Machine', 'Cardio', 'Bands', 'Other']
    
    for btn, choice in zip(btns, choices):
        btn.click(
            fn=lambda i, c=choice: process_choice(c, i), 
            inputs=current_id, 
            outputs=[current_id, exercise_display, stats_display, detailed_stats_display, image_window]
        )

demo.launch(debug=True)

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


IMPORTANT: You are using gradio version 3.41.2, however version 4.44.1 is available, please upgrade.
--------


In [15]:
import pandas as pd

# 1. Загружаем
manual_df = pd.read_csv('./exercises_to_review_MANUAL.csv')
reviewed_df = pd.read_csv('./reviewed_exercises.csv')

# 2. Чистим типы
manual_df['candidate_id'] = manual_df['candidate_id'].astype(str)
reviewed_df['candidate_id'] = reviewed_df['candidate_id'].astype(str)

# 3. Мерджим
# Мы берем manual_df за основу
final_mapping = manual_df.merge(
    reviewed_df[['candidate_id', 'equipment_golden']], 
    on='candidate_id', 
    how='left',
    suffixes=('_old', '_new')
)

# 4. ЛОГИКА ЗАПОЛНЕНИЯ:
# Если есть новое значение из Gradio - берем его. 
# Если нет (NaN) - берем то, что было в MANUAL (Bodyweight, Other и т.д.)
final_mapping['equipment_golden'] = final_mapping['equipment_golden_new'].fillna(final_mapping['equipment_golden_old'])

# 5. Оставляем только нужные колонки
final_mapping = final_mapping[['candidate_id', 'exercise_name', 'equipment_golden']]

# 6. Убираем те, что остались REVIEW_NEEDED (если ты их не прокликал)
# Или оставляем как есть, если хочешь видеть фронт работ
final_mapping.to_csv('./id_to_equipment_mapping_FIXED.csv', index=False)

print("--- Исправлено! ---")
print(final_mapping.head(5)) # Проверь первые строки, там теперь должны быть Bodyweight и Tire Flip

--- Исправлено! ---
  candidate_id              exercise_name equipment_golden
0         1020            Garhammer Raise       Bodyweight
1         2791                  Tire Flip            Other
2         2433  Single Arm Shoulder Press         Dumbbell
3         1577           Lying Bicep Curl         Dumbbell
4          873      Eccentric Calf Raises       Bodyweight


In [16]:
main_df = pd.read_csv('./rank_dataset_98k.csv')
main_df['candidate_id'] = main_df['candidate_id'].astype(str)

# Мерджим наш новый FIXED файл
main_df = main_df.merge(
    final_mapping[['candidate_id', 'equipment_golden']], 
    on='candidate_id', 
    how='left'
)

main_df.rename(columns={'equipment_golden': 'eq_clean'}, inplace=True)
main_df['eq_clean'] = main_df['eq_clean'].fillna('unlabeled')

# Важно: если в eq_clean осталось "REVIEW_NEEDED", заменим на "unlabeled"
main_df.loc[main_df['eq_clean'] == 'REVIEW_NEEDED', 'eq_clean'] = 'unlabeled'

main_df.to_csv('./rank_dataset_98k_with_labels.csv', index=False)
print("Готово! Проверь результат.")

Готово! Проверь результат.


In [17]:
main_df

,bert_score,candidate_id,level,goal,equipment,is_same_group,target,eq_clean
0,0.001939,1020,"['Novice', 'Beginner']",['Bodybuilding'],At Home,0,0,Bodyweight
1,0.001857,2784,"['Advanced', 'Intermediate', 'Beginner', 'Novi...","['Powerlifting', 'Powerbuilding']",Full Gym,0,0,Bodyweight
2,0.001643,297,"['Beginner', 'Novice', 'Intermediate']","['Powerbuilding', 'Bodybuilding']",Full Gym,0,0,Bodyweight
3,0.001590,2742,['Advanced'],['Bodybuilding'],Full Gym,0,0,Bodyweight
4,0.001532,834,['Intermediate'],"['Bodybuilding', 'Muscle & Sculpting', 'Athlet...",Full Gym,1,0,Dumbbell
...,...,...,...,...,...,...,...,...
9923555,0.001318,290,"['Novice', 'Beginner', 'Intermediate', 'Advanc...",['Bodybuilding'],Garage Gym,0,0,Machine
9923556,0.001264,1783,['Beginner'],['Powerlifting'],Garage Gym,0,0,Barbell
9923557,0.001263,923,"['Intermediate', 'Advanced']",['Powerlifting'],Full Gym,0,0,Barbell
9923558,0.001169,108,['Intermediate'],"['Bodybuilding', 'Powerbuilding']",Full Gym,1,0,Other


In [20]:
import pandas as pd
import re

print("📥 Загрузка данных...")
# Загружаем наш файл с eq_clean
df = pd.read_csv('./rank_dataset_98k_with_labels.csv')

# ==========================================
# 1. ОБРАБОТКА LEVEL (Сложность)
# ==========================================
print("🔄 Преобразование Level...")
level_hierarchy = {
    'Novice': 0,
    'Beginner': 1,
    'Intermediate': 2,
    'Advanced': 3
}

def collapse_to_min_level(text):
    if not isinstance(text, str): 
        return 'Beginner'
    
    found_levels = []
    for level_name in level_hierarchy.keys():
        if level_name.lower() in text.lower():
            found_levels.append(level_name)
    
    if not found_levels:
        return 'Beginner'
    
    # Сортируем и берем самый легкий уровень
    found_levels.sort(key=lambda x: level_hierarchy[x])
    return found_levels[0]

# Создаем level_clean, а затем сразу мапим его в индексы (0, 1, 2, 3)
df['level_clean'] = df['level'].apply(collapse_to_min_level)
level_map = {'Novice': 0, 'Beginner': 1, 'Intermediate': 2, 'Advanced': 3}
df['level_idx'] = df['level_clean'].map(level_map)


# ==========================================
# 2. ОБРАБОТКА GOAL (Цель тренировки)
# ==========================================
print("🎯 Преобразование Goal...")
goal_priority = [
    'OlympicWeightlifting', 'BodyweightFitness', 'Fitness', 
    'Powerlifting', 'Powerbuilding', 'Athletics', 'Muscle & Sculpting', 'Muscle&Sculpting', 'Bodybuilding'
]

def get_golden_goal(text):
    # Если вдруг пришел float(NaN) или None
    if not isinstance(text, str): 
        return 'Fitness'
    
    # Аккуратно убираем только квадратные скобки и кавычки (одинарные и двойные)
    # Пробелы не трогаем, чтобы не сломать "Muscle & Sculpting"
    clean_text = re.sub(r"[\[\]'\"]", "", text)
    
    # Разбиваем по запятой и убираем лишние пробелы по краям
    found_goals = [g.strip() for g in clean_text.split(',') if g.strip()]
    
    # Ищем цель с наивысшим приоритетом
    for p in goal_priority:
        if p in found_goals: 
            return str(p) # Возвращаем строго как строку
            
    # Если ничего из приоритетов не совпало, берем первую цель
    if found_goals:
        return str(found_goals[0])
        
    return 'Fitness'

# Применяем функцию и жестко фиксируем тип колонки как string
df['goal_clean'] = df['goal'].apply(get_golden_goal).astype(str)


# ==========================================
# 3. ФИНАЛЬНАЯ ЧИСТКА И СОХРАНЕНИЕ
# ==========================================
print("💾 Сохранение результата...")

# Удаляем сырые текстовые колонки, так как у нас теперь есть их чистые версии
cols_to_drop = ['level', 'goal', 'level_clean', 'equipment']
existing_cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=existing_cols_to_drop, inplace=True)

# Сохраняем как финальный датасет для обучения
final_filename = './rank_dataset_FINAL_READY.csv'
df.to_csv(final_filename, index=False)

print("✅ Готово! Датасет полностью подготовлен для CatBoost.")
print(df.head(3))

📥 Загрузка данных...
🔄 Преобразование Level...
🎯 Преобразование Goal...
💾 Сохранение результата...
✅ Готово! Датасет полностью подготовлен для CatBoost.
   bert_score  candidate_id  is_same_group  target    eq_clean  level_idx  \
0    0.001939          1020              0       0  Bodyweight          0   
1    0.001857          2784              0       0  Bodyweight          0   
2    0.001643           297              0       0  Bodyweight          0   

      goal_clean  
0   Bodybuilding  
1   Powerlifting  
2  Powerbuilding  


In [1]:
df

NameError: name 'df' is not defined

In [22]:
df['goal_clean'].unique()

array(['Bodybuilding', 'Powerlifting', 'Powerbuilding', 'Athletics',
       'Muscle & Sculpting', 'Fitness', 'Bodyweight Fitness'],
      dtype=object)

In [23]:
import pandas as pd

# 1. Загрузка исходного размеченного датасета
df = pd.read_csv('./rank_dataset_FINAL_READY.csv')

# 2. Логика маппинга локаций
def assign_location(row):
    eq = row['eq_clean']
    
    # Outdoor — только свой вес (самый строгий фильтр)
    if eq == 'Bodyweight':
        return 'Outdoor'
    
    # Home — свой вес, резинки и гантели (мобильный инвентарь)
    if eq in ['Bodyweight', 'Bands', 'Dumbbell']:
        return 'Home'
    
    # Gym — штанги и тренажеры (стационарное оборудование)
    if eq in ['Barbell', 'Machine']:
        return 'Gym'
    
    return 'Gym' # По умолчанию отправляем в зал, если тип неясен

print("🌍 Группировка упражнений по локациям...")
df['location'] = df.apply(assign_location, axis=1)

# 3. Важный фикс: Дублируем строки для Home и Gym
# Упражнение со своим весом (Outdoor) доступно и ДОМА, и в ЗАЛЕ.
# Упражнение с гантелями (Home) доступно и в ЗАЛЕ.

def expand_locations(df):
    new_rows = []
    for _, row in df.iterrows():
        loc = row['location']
        
        # Если упражнение Outdoor (Bodyweight), оно доступно ВЕЗДЕ
        if loc == 'Outdoor':
            for l in ['Outdoor', 'Home', 'Gym']:
                new_row = row.copy()
                new_row['location'] = l
                new_rows.append(new_row)
        
        # Если упражнение Home (Dumbbell/Bands), оно доступно и в ЗАЛЕ
        elif loc == 'Home':
            for l in ['Home', 'Gym']:
                new_row = row.copy()
                new_row['location'] = l
                new_rows.append(new_row)
        
        # Если упражнение Gym (Barbell/Machine), оно только в ЗАЛЕ
        else:
            new_rows.append(row)
            
    return pd.DataFrame(new_rows)

print("📈 Расширение доступности упражнений (Outdoor -> Home -> Gym)...")
df_expanded = expand_locations(df)

# Сохраняем финальный датасет для обучения
df_expanded.to_csv('./rank_dataset_LOCATION_FINAL.csv', index=False)

print("\n✅ Новый датасет готов!")
print("Распределение после расширения:")
print(df_expanded['location'].value_counts())

🌍 Группировка упражнений по локациям...
📈 Расширение доступности упражнений (Outdoor -> Home -> Gym)...

✅ Новый датасет готов!
Распределение после расширения:
location
Gym        9923560
Home       3729539
Outdoor    1908190
Name: count, dtype: int64
